# Test: 100k Run Postprocessing (step-by-step)

Libreta para depurar `100k_run_postprocessing_parallel.py` paso a paso.

**Supuesto clave:** `tmp/uganda_0.csv` ya existe con el baseline (`primary_id == 0`).
No se hace ningún renombramiento — el baseline siempre es `primary_id == 0`.

**Flujo:**
1. Configuración y paths
2. Pre-checks (archivos locales + baseline CSV)
3. Conexión S3 y descarga de datos
4. Inspeccionar `attribute_primary_df` y lista de `primary_ids`
5. Cachear datos localmente
6. Init R + función `rescale`
7. Descomposición para UN primary_id no-baseline
8. Preparar columnas y upload de emisiones
9. CBA: append baseline + actual, calcular, subir
10. Cleanup

---
## 0. Imports y environment

In [136]:
import os, sys, traceback, shutil
from io import StringIO

import pandas as pd
import numpy as np
import yaml
import boto3

os.environ.setdefault("OMP_NUM_THREADS", "1")
os.environ.setdefault("OPENBLAS_NUM_THREADS", "1")
os.environ.setdefault("MKL_NUM_THREADS", "1")
os.environ.setdefault("NUMEXPR_NUM_THREADS", "1")

print("Imports OK")

Imports OK


---
## 1. Configuración de paths y parámetros

In [137]:
# ── Ajusta si es necesario ──────────────────────────────────
DIR_ID          = "130"
RUN_ID          = "sisepuede_run_2026-03-10t13;27;53.264959"
TARGET_COUNTRY  = "UGA"
TIME_PERIOD_REF = 4       # igual que en el script (2015 + 7 = 2022)
PRIMARY_ID_BASE = 0       # siempre 0
# ────────────────────────────────────────────────────────────

SCRIPT_DIR    = os.path.abspath(".")
CONFIG_DIR    = os.path.join(SCRIPT_DIR, "config")
CW_DIR        = os.path.join(SCRIPT_DIR, "cw")
TMP_DIR       = os.path.join(SCRIPT_DIR, "tmp")
R_SCRIPTS_DIR = os.path.join(SCRIPT_DIR, "r_scripts")
CACHE_DIR     = os.path.join(TMP_DIR, "cache")

os.makedirs(TMP_DIR, exist_ok=True)
os.makedirs(CACHE_DIR, exist_ok=True)

AWS_CONFIG_PATH       = os.path.join(CONFIG_DIR, "aws_credentials_config.yaml")
EMISSION_TARGETS_PATH = os.path.join(CW_DIR, "emission_targets_uganda_2019_LULUCF.csv")
R_SCRIPT_PATH         = os.path.join(R_SCRIPTS_DIR, "intertemporal_decomposition.r")
CB_CONFIG_PATH        = os.path.join(CONFIG_DIR, "cb_config_params.xlsx")
BASELINE_CSV_PATH     = os.path.join(TMP_DIR, "uganda_0.csv")

print("Paths configurados")
print(f"  TMP_DIR  : {TMP_DIR}")
print(f"  CACHE_DIR: {CACHE_DIR}")

Paths configurados
  TMP_DIR  : /Users/fabianfuentes/git/ssp_uganda_data/ssp_modeling/100k_runs_postprocessing/tmp
  CACHE_DIR: /Users/fabianfuentes/git/ssp_uganda_data/ssp_modeling/100k_runs_postprocessing/tmp/cache


---
## 2. Pre-checks — verificar todos los archivos antes de empezar

In [138]:
required_files = {
    "aws_credentials_config.yaml" : AWS_CONFIG_PATH,
    "emission_targets.csv"        : EMISSION_TARGETS_PATH,
    "intertemporal_decomposition.r": R_SCRIPT_PATH,
    "cb_config_params.xlsx"       : CB_CONFIG_PATH,
    "uganda_0.csv (BASELINE)"     : BASELINE_CSV_PATH,   # pre-condición crítica
}

all_ok = True
for name, path in required_files.items():
    exists = os.path.exists(path)
    status = "OK" if exists else "MISSING"
    if not exists:
        all_ok = False
    print(f"[{status}] {name}")
    if not exists:
        print(f"        -> {path}")

if not all_ok:
    print("\nFaltan archivos. Corrige antes de continuar.")
else:
    print("\nTodos los archivos presentes. Listo para continuar.")

[OK] aws_credentials_config.yaml
[OK] emission_targets.csv
[OK] intertemporal_decomposition.r
[OK] cb_config_params.xlsx
[OK] uganda_0.csv (BASELINE)

Todos los archivos presentes. Listo para continuar.


In [139]:
# Inspeccionar el baseline CSV
baseline_df = pd.read_csv(BASELINE_CSV_PATH)
print(f"uganda_0.csv shape : {baseline_df.shape}")
print(f"primary_id únicos  : {baseline_df['primary_id'].unique()}")
baseline_df.head(2)

uganda_0.csv shape : (56, 4054)
primary_id únicos  : [0]


,primary_id,region,time_period,area_gnrl_country_ha,area_lndu_infimum_croplands_ha,area_lndu_infimum_flooded_ha,area_lndu_infimum_forests_mangroves_ha,area_lndu_infimum_forests_primary_ha,area_lndu_infimum_forests_secondary_ha,area_lndu_infimum_grasslands_ha,...,yield_agrc_fruits_tonne,yield_agrc_herbs_and_other_perennial_crops_tonne,yield_agrc_nuts_tonne,yield_agrc_other_annual_tonne,yield_agrc_other_woody_perennial_tonne,yield_agrc_pulses_tonne,yield_agrc_rice_tonne,yield_agrc_sugar_cane_tonne,yield_agrc_tubers_tonne,yield_agrc_vegetables_and_vines_tonne
0,0,uganda,0,24155000,-999,3526780,-999,146461.012134,383133.987866,-999,...,8.162759e+06,1049.688971,382983.417459,2.802339e+06,1294.273957,710613.538152,308317.181306,6.763261e+06,7.028016e+06,2.357715e+06
1,0,uganda,1,24155000,-999,3526780,-999,146461.012134,383133.987866,-999,...,8.014876e+06,1077.840167,361439.962580,2.819014e+06,1299.678998,731167.999372,317530.631920,6.850348e+06,7.235665e+06,2.408238e+06


---
## 3. Conexión a S3 y construcción de prefijos

In [140]:
def read_yaml(file_path):
    with open(file_path, 'r') as f:
        return yaml.safe_load(f)

aws_config   = read_yaml(AWS_CONFIG_PATH)
PROFILE_NAME = aws_config["profile_name"]
BUCKET_NAME  = aws_config["bucket_name"]
print(f"Profile : {PROFILE_NAME}  |  Bucket : {BUCKET_NAME}")

session = boto3.Session(profile_name=PROFILE_NAME)
s3      = session.resource('s3')

try:
    s3.meta.client.head_bucket(Bucket=BUCKET_NAME)
    print("Conexión S3: OK")
except Exception as e:
    print(f"ERROR S3: {e}")

Profile : fabian  |  Bucket : sisepuede-data
Conexión S3: OK


In [141]:
RUN_DB_PREFIX        = f'run_database/{RUN_ID}/'
MODEL_OUTPUT_PREFIX  = f'{RUN_DB_PREFIX}model_output/region=uganda/model_output_{DIR_ID}/'
MODEL_INPUT_PREFIX   = f'{RUN_DB_PREFIX}model_input/region=uganda/model_input_{DIR_ID}/'
TRANSFER_PREFIX      = f'transfers/{RUN_ID}/'
S3_DECOMPOSED_PREFIX = f'{RUN_DB_PREFIX}decomposed_outputs/'
S3_CB_PREFIX         = f'{RUN_DB_PREFIX}cb_outputs/'
S3_JOBS_PREFIX       = f'{RUN_DB_PREFIX}jobs_outputs/'

print("MODEL_OUTPUT :", MODEL_OUTPUT_PREFIX)
print("MODEL_INPUT  :", MODEL_INPUT_PREFIX)
print("TRANSFER     :", TRANSFER_PREFIX)

MODEL_OUTPUT : run_database/sisepuede_run_2026-03-10t13;27;53.264959/model_output/region=uganda/model_output_130/
MODEL_INPUT  : run_database/sisepuede_run_2026-03-10t13;27;53.264959/model_input/region=uganda/model_input_130/
TRANSFER     : transfers/sisepuede_run_2026-03-10t13;27;53.264959/


---
## 4. Descarga de datos desde S3

In [142]:
def fetch_csv_from_s3(s3_resource, bucket_name, key):
    obj     = s3_resource.Object(bucket_name, key)
    content = obj.get()['Body'].read().decode('utf-8')
    return pd.read_csv(StringIO(content))

print("Descargando output_df...")
output_df = fetch_csv_from_s3(s3, BUCKET_NAME, f'{MODEL_OUTPUT_PREFIX}data.csv')
print(f"  shape: {output_df.shape}")
output_df.head(2)

Descargando output_df...
  shape: (2184, 1637)


,primary_id,region,time_period,area_agrc_crops_bevs_and_spices,area_agrc_crops_cereals,area_agrc_crops_fibers,area_agrc_crops_fruits,area_agrc_crops_herbs_and_other_perennial_crops,area_agrc_crops_nuts,area_agrc_crops_other_annual,...,yield_agrc_fruits_tonne,yield_agrc_herbs_and_other_perennial_crops_tonne,yield_agrc_nuts_tonne,yield_agrc_other_annual_tonne,yield_agrc_other_woody_perennial_tonne,yield_agrc_pulses_tonne,yield_agrc_rice_tonne,yield_agrc_sugar_cane_tonne,yield_agrc_tubers_tonne,yield_agrc_vegetables_and_vines_tonne
0,6520974,uganda,0,621061.398901,1.981597e+06,152656.721347,1.107040e+06,494.365315,502585.165893,2.851566e+06,...,8.162759e+06,1049.688971,382983.417459,2.802339e+06,1294.273957,710613.538152,308317.181306,6.763261e+06,7.028016e+06,2.357715e+06
1,6520974,uganda,1,623655.024739,1.989872e+06,153294.233866,1.111663e+06,496.429843,504684.020973,2.863475e+06,...,8.014876e+06,1077.840167,361439.962580,2.819014e+06,1299.678998,731167.999372,317530.631920,6.850348e+06,7.235665e+06,2.408238e+06


In [143]:
print("Descargando input_df...")
input_df = fetch_csv_from_s3(s3, BUCKET_NAME, f'{MODEL_INPUT_PREFIX}data.csv')
print(f"  shape: {input_df.shape}")
input_df.head(2)

Descargando input_df...
  shape: (2184, 2420)


,primary_id,region,time_period,area_gnrl_country_ha,area_lndu_infimum_croplands_ha,area_lndu_infimum_flooded_ha,area_lndu_infimum_forests_mangroves_ha,area_lndu_infimum_forests_primary_ha,area_lndu_infimum_forests_secondary_ha,area_lndu_infimum_grasslands_ha,...,yf_agrc_herbs_and_other_perennial_crops_tonne_ha,yf_agrc_nuts_tonne_ha,yf_agrc_other_annual_tonne_ha,yf_agrc_other_woody_perennial_tonne_ha,yf_agrc_pulses_tonne_ha,yf_agrc_rice_tonne_ha,yf_agrc_sugar_cane_tonne_ha,yf_agrc_tubers_tonne_ha,yf_agrc_vegetables_and_vines_tonne_ha,yf_lndu_supremum_pastures_tonne_per_ha
0,6520974,uganda,0,24155000.0,-999.0,3526780.0,-999.0,146461.012134,383133.987866,-999.0,...,2.123306,0.762027,0.982737,0.3674,0.784492,2.755775,76.794300,4.446020,4.002223,92.81
1,6520974,uganda,1,24155000.0,-999.0,3526780.0,-999.0,146461.012134,383133.987866,-999.0,...,2.171183,0.716171,0.984473,0.3674,0.803826,2.826323,77.459654,4.558346,4.070985,92.81


In [144]:
print("Descargando attribute tables...")
attribute_primary_df  = fetch_csv_from_s3(s3, BUCKET_NAME, f'{TRANSFER_PREFIX}ATTRIBUTE_PRIMARY.csv')
attribute_strategy_df = fetch_csv_from_s3(s3, BUCKET_NAME, f'{TRANSFER_PREFIX}ATTRIBUTE_STRATEGY.csv')
print(f"  attribute_primary  : {attribute_primary_df.shape}")
print(f"  attribute_strategy : {attribute_strategy_df.shape}")
attribute_primary_df.head(3)

Descargando attribute tables...
  attribute_primary  : (20124, 4)
  attribute_strategy : (194, 6)


,primary_id,design_id,strategy_id,future_id
0,0,0,0,0
1,700070,0,6004,0
2,740074,0,6008,0


In [145]:
emission_targets_df = pd.read_csv(EMISSION_TARGETS_PATH)
print(f"  emission_targets : {emission_targets_df.shape}")
emission_targets_df.head(2)

  emission_targets : (54, 7)


,ssp_subsector,Subsector,Gas,Vars,est_from_sisepuede,Subsector_Category,UGA
0,agrc,Agriculture and Managed Soil,ch4,emission_co2e_ch4_agrc_biomass_burning:emissio...,0,Agriculture and Managed Soil:ch4,0.241921
1,agrc,Agriculture and Managed Soil,co2,emission_co2e_co2_agrc_biomass_bevs_and_spices...,0,Agriculture and Managed Soil:co2,0.025526


---
## 5. Inspeccionar `primary_ids` y confirmar que el baseline (0) no está en output_df

In [146]:
all_primary_ids = sorted(output_df["primary_id"].dropna().astype(int).unique().tolist())
print(f"Total primary_ids en output_df: {len(all_primary_ids)}")
print(f"Primeros 10: {all_primary_ids[:10]}")
print(f"primary_id=0 en output_df    : {0 in all_primary_ids}  (esperado: False — baseline está en disco)")

# primary_ids a procesar: excluir 0
primary_ids_to_process = [pid for pid in all_primary_ids if pid != PRIMARY_ID_BASE]
print(f"\nA procesar (sin baseline): {len(primary_ids_to_process)}")

Total primary_ids en output_df: 39
Primeros 10: [6520974, 6520975, 6520976, 6520977, 6520978, 6520979, 6520980, 6520981, 6520982, 6520983]
primary_id=0 en output_df    : False  (esperado: False — baseline está en disco)

A procesar (sin baseline): 39


---
## 6. Cachear datos localmente

In [147]:
output_df.to_pickle(os.path.join(CACHE_DIR, "output_df.pkl"))
input_df.to_pickle(os.path.join(CACHE_DIR, "input_df.pkl"))
attribute_primary_df.to_pickle(os.path.join(CACHE_DIR, "attribute_primary_df.pkl"))
attribute_strategy_df.to_pickle(os.path.join(CACHE_DIR, "attribute_strategy_df.pkl"))
emission_targets_df.to_pickle(os.path.join(CACHE_DIR, "emission_targets_df.pkl"))

# Baseline cacheado separado: ya tiene input+output merged, primary_id=0
# No se appenda a output_df para no mezclar schemas (output_df es lo que va a R)
baseline_df.to_pickle(os.path.join(CACHE_DIR, "baseline_decomposed_df.pkl"))

print(f"Cache guardado en: {CACHE_DIR}")
print(f"  output_df           : {output_df.shape}")
print(f"  baseline_decomposed : {baseline_df.shape}  ← input+output ya mergeados")

Cache guardado en: /Users/fabianfuentes/git/ssp_uganda_data/ssp_modeling/100k_runs_postprocessing/tmp/cache
  output_df           : (2184, 1637)
  baseline_decomposed : (56, 4054)  ← input+output ya mergeados


---
## 7. Inicializar rpy2 y cargar función R `rescale`

In [148]:
import rpy2.robjects as ro
from rpy2.robjects import pandas2ri, default_converter
from rpy2.robjects.conversion import localconverter

print(f"Cargando: {R_SCRIPT_PATH}")
ro.r['source'](R_SCRIPT_PATH)
r_rescale = ro.globalenv['rescale']
print("Función 'rescale' cargada OK")

# Verificar firma — debe tener exactamente 7 parámetros (sin r_run)
params = list(ro.r("names(formals(rescale))"))
print(f"Parámetros ({len(params)}): {params}")

Cargando: /Users/fabianfuentes/git/ssp_uganda_data/ssp_modeling/100k_runs_postprocessing/r_scripts/intertemporal_decomposition.r
Función 'rescale' cargada OK
Parámetros (7): ['z', 'rall', 'data_all', 'te_all', 'initial_conditions_id', 'dir.output', 'time_period_ref']


---
## 8. Helpers

In [149]:
def sanitize_for_r(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    def _is_scalar(x):
        return not isinstance(x, (list, dict, pd.Series))
    for c in df.columns:
        s = df[c]
        if not s.map(_is_scalar).all():
            df[c] = s.astype(str)
        elif s.dtype == "object":
            df[c] = df[c].astype("string")
    return df

def upload_df_to_s3(df, s3_resource, bucket, key):
    buf = StringIO()
    df.to_csv(buf, index=False)
    s3_resource.Object(bucket, key).put(Body=buf.getvalue(), ContentType="text/csv")
    print(f"Uploaded -> s3://{bucket}/{key}")

print("Helpers definidos")

Helpers definidos


---
## 9. Elegir primary_id de prueba (no-baseline)

In [150]:
# Cambia este valor para probar con un primary_id específico
TEST_PRIMARY_ID = primary_ids_to_process[0]
print(f"primary_id de prueba: {TEST_PRIMARY_ID}")

# Verificar que tiene mapeo en attribute_primary_df
row = attribute_primary_df.loc[attribute_primary_df["primary_id"] == TEST_PRIMARY_ID]
print(row)

primary_id de prueba: 6520974
       primary_id  design_id  strategy_id  future_id
10445     6520974          3         6004        322


---
## 10. Descomposición R para TODOS los primary_ids

`data_all` se construye mergeando `output_df` + `input_df` para **todos** los `primary_id`
y se agrega el baseline (primary_id=0). Esto permite que el output de `rescale()` ya tenga
input+output listo para CB sin ningún merge adicional.

In [151]:
# --- 10a. Filtrar y validar ---
data_all = output_df.loc[output_df["primary_id"] == TEST_PRIMARY_ID].copy()

print(f"Filas para primary_id={TEST_PRIMARY_ID}: {len(data_all)}")

if data_all.empty:
    print("ERROR: no hay filas para este primary_id")
else:
    # Agregar baseline (primary_id=0) como referencia para R.
    # baseline_df tiene input+output merged — solo usamos las columnas de output_df
    # para mantener el mismo schema que data_all.
    baseline_output_cols = [c for c in output_df.columns if c in baseline_df.columns]
    baseline_for_r = baseline_df[baseline_output_cols].copy()
    data_all = pd.concat([baseline_for_r, data_all], ignore_index=True)

    data_all = data_all.fillna(0)
    print(f"Filas tras append baseline: {len(data_all)}")
    print(f"primary_ids en data_all   : {sorted(data_all['primary_id'].unique().tolist())}")
    print(f"  → esperado: [0, {TEST_PRIMARY_ID}]")
    for col in ("region", "primary_id", "time_period"):
        status = "OK" if col in data_all.columns else "MISSING"
        print(f"  [{status}] columna '{col}'")

Filas para primary_id=6520974: 56
Filas tras append baseline: 112
primary_ids en data_all   : [0, 6520974]
  → esperado: [0, 6520974]
  [OK] columna 'region'
  [OK] columna 'primary_id'
  [OK] columna 'time_period'


In [152]:
# --- 10b. Filtrar por time_period (TIME_PERIOD_REF = 7) ---
data_all = data_all.loc[data_all["time_period"] >= TIME_PERIOD_REF].copy()
print(f"Filas tras filtrar time_period >= {TIME_PERIOD_REF}: {len(data_all)}")
if data_all.empty:
    print("ERROR: todos los registros fueron filtrados")

Filas tras filtrar time_period >= 4: 104


In [153]:
baseline_for_r.shape

(56, 1637)

In [154]:
data_all

,primary_id,region,time_period,area_agrc_crops_bevs_and_spices,area_agrc_crops_cereals,area_agrc_crops_fibers,area_agrc_crops_fruits,area_agrc_crops_herbs_and_other_perennial_crops,area_agrc_crops_nuts,area_agrc_crops_other_annual,...,yield_agrc_fruits_tonne,yield_agrc_herbs_and_other_perennial_crops_tonne,yield_agrc_nuts_tonne,yield_agrc_other_annual_tonne,yield_agrc_other_woody_perennial_tonne,yield_agrc_pulses_tonne,yield_agrc_rice_tonne,yield_agrc_sugar_cane_tonne,yield_agrc_tubers_tonne,yield_agrc_vegetables_and_vines_tonne
4,0,uganda,4,634960.593061,2.025945e+06,156073.139455,1.131815e+06,505.429083,5.138329e+05,2.915383e+06,...,8.170898e+06,1097.824878,2.353971e+05,4.455654e+06,1323.239475,7.407067e+05,3.397411e+05,6.998599e+06,7.766890e+06,2.430411e+06
5,0,uganda,5,640126.452982,2.042427e+06,157342.906406,1.141024e+06,509.541112,5.180133e+05,2.939102e+06,...,8.218131e+06,1119.356653,3.166257e+05,4.942803e+06,1334.004978,7.521050e+05,3.496743e+05,7.082474e+06,7.234120e+06,2.436637e+06
6,0,uganda,6,642767.262368,2.050853e+06,157992.016628,1.145731e+06,511.643198,5.201503e+05,2.951227e+06,...,8.245762e+06,1131.174767,4.376510e+05,6.769841e+06,1339.508348,7.569232e+05,3.567074e+05,5.599249e+06,8.034844e+06,2.453034e+06
7,0,uganda,7,645322.479649,2.059006e+06,158620.088334,1.150285e+06,513.677153,5.222181e+05,2.962959e+06,...,8.272244e+06,1143.070327,2.093050e+05,7.998025e+06,1344.833347,7.624223e+05,3.597913e+05,7.069706e+06,8.088089e+06,2.466888e+06
8,0,uganda,8,647798.760608,2.066907e+06,159228.757514,1.154699e+06,515.648274,5.242220e+05,2.974329e+06,...,8.303987e+06,1147.456604,2.101081e+05,8.028716e+06,1349.993844,7.653479e+05,3.611719e+05,7.096834e+06,8.119125e+06,2.476354e+06
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
107,6520974,uganda,51,862551.556353,2.285799e+06,138019.279964,1.697993e+06,352.765002,1.420008e+06,1.468909e+06,...,3.571618e+07,2296.039204,1.664676e+06,1.159747e+07,1773.522848,2.901170e+06,1.097548e+06,2.053784e+07,3.077376e+07,1.031455e+07
108,6520974,uganda,52,872355.029763,2.306445e+06,139376.872120,1.717950e+06,349.538109,1.437636e+06,1.481404e+06,...,3.628479e+07,2284.407202,1.692283e+06,1.174430e+07,1774.623794,2.945862e+06,1.113309e+06,2.081441e+07,3.125747e+07,1.047824e+07
109,6520974,uganda,53,882096.050920,2.327573e+06,140745.539836,1.737526e+06,346.482441,1.454750e+06,1.494431e+06,...,3.684832e+07,2273.696152,1.719431e+06,1.189601e+07,1776.934165,2.990207e+06,1.128996e+06,2.109182e+07,3.173800e+07,1.064056e+07
110,6520974,uganda,54,891765.338741,2.349120e+06,142118.643260,1.756743e+06,343.578716,1.471405e+06,1.507839e+06,...,3.740719e+07,2263.799349,1.746180e+06,1.205150e+07,1780.248777,3.034216e+06,1.144590e+06,2.136923e+07,3.221552e+07,1.080161e+07


In [155]:
# --- 10c. Construir te_df con las columnas que usa R ---
# El script usa ["Subsector","Gas","Vars","Edgar_Class", target_country]
# Verificar si la columna existe, si no mostrar las disponibles
print(f"Columnas de emission_targets_df: {emission_targets_df.columns.tolist()}")

te_cols = ["Subsector","Gas","Vars","Subsector_Category","ssp_subsector", TARGET_COUNTRY]
missing_te = [c for c in te_cols if c not in emission_targets_df.columns]
if missing_te:
    print(f"WARNING columnas faltantes en te_df: {missing_te}")
    print("Columnas disponibles:", emission_targets_df.columns.tolist())

te_df = emission_targets_df[[c for c in te_cols if c in emission_targets_df.columns]].copy()
te_df = te_df.rename(columns={TARGET_COUNTRY: "tvalue"})
te_df = sanitize_for_r(te_df)

with localconverter(default_converter + pandas2ri.converter):
    r_data_all = ro.conversion.py2rpy(sanitize_for_r(data_all))
with localconverter(default_converter + pandas2ri.converter):
    r_te_all = ro.conversion.py2rpy(te_df)

rall = data_all["region"].dropna().astype(str).unique().tolist()
print(f"Regiones: {rall}")
print(f"te_df shape: {te_df.shape}  columnas: {te_df.columns.tolist()}")

Columnas de emission_targets_df: ['ssp_subsector', 'Subsector', 'Gas', 'Vars', 'est_from_sisepuede', 'Subsector_Category', 'UGA']
Regiones: ['uganda']
te_df shape: (54, 6)  columnas: ['Subsector', 'Gas', 'Vars', 'Subsector_Category', 'ssp_subsector', 'tvalue']


In [156]:
# --- 10c.5 Pre-flight: reemplazar ceros en variables de emisión en time_period_ref ---
# Equivalente Python del bloque R que corre justo antes de llamar r_rescale().
# Modifica data_all in-place y re-genera r_data_all.

_EXCLUDE_VARS = {
    "emission_co2e_co2_ccsq_direct_air_capture",
    "emission_co2e_ch4_ccsq_direct_air_capture",
    "emission_co2e_n2o_ccsq_direct_air_capture",
}

# Extraer variables únicas de te_df["Vars"] (separadas por ":")
_all_vars = set()
for _v in te_df["Vars"].dropna():
    for _part in str(_v).split(":"):
        _part = _part.strip()
        if _part:
            _all_vars.add(_part)
_all_vars -= _EXCLUDE_VARS

# Revisar cobertura y reemplazar ceros en time_period == TIME_PERIOD_REF
_missing_vars = [v for v in sorted(_all_vars) if v not in data_all.columns]
if _missing_vars:
    print(f"WARNING: {len(_missing_vars)} columna(s) no encontrada(s) en data_all:")
    for _m in _missing_vars:
        print(f"  MISSING: {_m}")

for _var in sorted(_all_vars):
    if _var not in data_all.columns:
        continue
    _mask = (data_all["time_period"] == TIME_PERIOD_REF) & (data_all[_var] == 0)
    _changed = int(_mask.sum())
    data_all.loc[_mask, _var] = 0.01
    if _changed > 0:
        print(f"Changed {_changed} zeros in: {_var} (time_period == {TIME_PERIOD_REF})")

# Re-convertir data_all a objeto R después de la modificación
with localconverter(default_converter + pandas2ri.converter):
    r_data_all = ro.conversion.py2rpy(sanitize_for_r(data_all))
print("r_data_all actualizado tras pre-flight check.")


Changed 2 zeros in: emission_co2e_ch4_entc_fuel_mining_and_extraction_me_coal (time_period == 4)
Changed 2 zeros in: emission_co2e_ch4_entc_fuel_mining_and_extraction_me_crude (time_period == 4)
Changed 2 zeros in: emission_co2e_ch4_entc_fuel_mining_and_extraction_me_natural_gas (time_period == 4)
Changed 2 zeros in: emission_co2e_ch4_entc_generation_pp_coal (time_period == 4)
Changed 2 zeros in: emission_co2e_ch4_entc_generation_pp_coal_ccs (time_period == 4)
Changed 2 zeros in: emission_co2e_ch4_entc_generation_pp_gas (time_period == 4)
Changed 2 zeros in: emission_co2e_ch4_entc_generation_pp_gas_ccs (time_period == 4)
Changed 2 zeros in: emission_co2e_ch4_entc_generation_pp_geothermal (time_period == 4)
Changed 2 zeros in: emission_co2e_ch4_entc_generation_pp_hydropower (time_period == 4)
Changed 2 zeros in: emission_co2e_ch4_entc_generation_pp_nuclear (time_period == 4)
Changed 2 zeros in: emission_co2e_ch4_entc_generation_pp_ocean (time_period == 4)
Changed 2 zeros in: emission_co

/var/folders/fj/jc6_dph55kg8wg_040b7gk980000gn/T/ipykernel_99552/2019562856.py:32: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '0.01' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  data_all.loc[_mask, _var] = 0.01


r_data_all actualizado tras pre-flight check.


In [157]:
data_all['primary_id'].unique()

array([      0, 6520974])

In [158]:
# --- 10d. Llamar R rescale ---
# rescale() acepta exactamente 7 argumentos: z, rall, data_all, te_all,
# initial_conditions_id, dir.output, time_period_ref
# NO hay r_run — R no acepta ese argumento.
import shutil

# Verificar aridad ANTES de llamar (falla rápido con mensaje claro)
_r_params = list(ro.r("names(formals(rescale))"))
assert len(_r_params) == 7, (
    f"Se esperan 7 parámetros en rescale(), se encontraron {len(_r_params)}: {_r_params}"
)
print(f"Verificación R OK: rescale() tiene {len(_r_params)} parámetros: {_r_params}")

worker_tmp = os.path.join(TMP_DIR, f"run_{TEST_PRIMARY_ID}")
os.makedirs(worker_tmp, exist_ok=True)

r_rall       = ro.StrVector([str(x) for x in rall])
r_init_ids = ro.StrVector(["_0"])
out_dir      = worker_tmp if worker_tmp.endswith(os.sep) else worker_tmp + os.sep
r_dir_output = ro.StrVector([out_dir])
r_z          = ro.IntVector([1])
r_time_ref   = ro.IntVector([TIME_PERIOD_REF])

# 7 args — exactamente lo que rescale() espera
print(f"Llamando r_rescale para primary_id={TEST_PRIMARY_ID}...")
print(f"Output dir: {out_dir}")
try:
    r_rescale(r_z, r_rall, r_data_all, r_te_all, r_init_ids, r_dir_output, r_time_ref)
    print("r_rescale OK")
except Exception as e:
    print(f"ERROR: {e}")
    traceback.print_exc()

Verificación R OK: rescale() tiene 7 parámetros: ['z', 'rall', 'data_all', 'te_all', 'initial_conditions_id', 'dir.output', 'time_period_ref']
Llamando r_rescale para primary_id=6520974...
Output dir: /Users/fabianfuentes/git/ssp_uganda_data/ssp_modeling/100k_runs_postprocessing/tmp/run_6520974/
[1] "uganda"
r_rescale OK


In [159]:
# --- 10e. Mover CSV de R al path canónico y limpiar worker_tmp ---
r_output   = os.path.join(worker_tmp, f"{rall[0]}.csv")   # e.g. run_1140114/uganda.csv

local_decomp = os.path.join(TMP_DIR, f"uganda_{TEST_PRIMARY_ID}.csv")

print(f"Archivo R generado : {r_output}  →  existe: {os.path.exists(r_output)}")
if os.path.exists(r_output):
    os.replace(r_output, local_decomp)
    shutil.rmtree(worker_tmp, ignore_errors=True)
    decomposed_df = pd.read_csv(local_decomp)
    print(f"OK: {local_decomp}  |  shape={decomposed_df.shape}")
    decomposed_df = decomposed_df[decomposed_df['primary_id'] != 0]
    display(decomposed_df.head())
else:
    print(f"ERROR: archivo no generado. Contenido de worker_tmp:")
    print(os.listdir(worker_tmp) if os.path.exists(worker_tmp) else "directorio no existe")

Archivo R generado : /Users/fabianfuentes/git/ssp_uganda_data/ssp_modeling/100k_runs_postprocessing/tmp/run_6520974/uganda.csv  →  existe: True
OK: /Users/fabianfuentes/git/ssp_uganda_data/ssp_modeling/100k_runs_postprocessing/tmp/uganda_6520974.csv  |  shape=(104, 1637)


,primary_id,region,time_period,area_agrc_crops_bevs_and_spices,area_agrc_crops_cereals,area_agrc_crops_fibers,area_agrc_crops_fruits,area_agrc_crops_herbs_and_other_perennial_crops,area_agrc_crops_nuts,area_agrc_crops_other_annual,...,yield_agrc_fruits_tonne,yield_agrc_herbs_and_other_perennial_crops_tonne,yield_agrc_nuts_tonne,yield_agrc_other_annual_tonne,yield_agrc_other_woody_perennial_tonne,yield_agrc_pulses_tonne,yield_agrc_rice_tonne,yield_agrc_sugar_cane_tonne,yield_agrc_tubers_tonne,yield_agrc_vegetables_and_vines_tonne
52,6520974,uganda,4,634960.593061,2.025945e+06,156073.139455,1.131815e+06,505.429083,513832.892470,2.915383e+06,...,8.170898e+06,1097.824878,235397.143197,4.455654e+06,1323.239475,740706.669054,339741.141725,6.998599e+06,7.766890e+06,2.430411e+06
53,6520974,uganda,5,640126.452982,2.042427e+06,157342.906406,1.141024e+06,509.541112,518013.291655,2.939102e+06,...,8.218131e+06,1119.356653,316625.727880,4.942803e+06,1334.004978,752105.027744,349674.336109,7.082474e+06,7.234120e+06,2.436637e+06
54,6520974,uganda,6,642767.262368,2.050853e+06,157992.016628,1.145731e+06,511.643198,520150.329354,2.951227e+06,...,8.245762e+06,1131.174767,437650.962060,6.769841e+06,1339.508348,756923.241373,356707.404945,5.599249e+06,8.034844e+06,2.453034e+06
55,6520974,uganda,7,645322.479649,2.059006e+06,158620.088334,1.150285e+06,513.677153,522218.102852,2.962959e+06,...,8.272244e+06,1143.070327,209304.971757,7.998025e+06,1344.833347,762422.283722,359791.316050,7.069706e+06,8.088089e+06,2.466888e+06
56,6520974,uganda,8,647798.760608,2.066907e+06,159228.757514,1.154699e+06,515.648274,524221.998246,2.974329e+06,...,8.303987e+06,1147.456604,210108.132862,8.028716e+06,1349.993844,765347.909039,361171.934909,7.096834e+06,8.119125e+06,2.476354e+06


In [160]:
# --- 10f. Merge decomposed + input_df ---
# Produce el mismo schema que el baseline (ambos: decomposed + input merged)
decomposed_df_merged = pd.merge(
    decomposed_df, input_df,
    on=["primary_id", "region", "time_period"],
    how="left"
)
print(f"decomposed_df_merged shape: {decomposed_df_merged.shape}")
print(f"primary_ids en merged     : {decomposed_df_merged['primary_id'].unique()}")
print(f"baseline shape            : {baseline_df.shape}  ← deben tener columnas compatibles")
decomposed_df_merged.head(2)

decomposed_df_merged shape: (52, 4054)
primary_ids en merged     : [6520974]
baseline shape            : (56, 4054)  ← deben tener columnas compatibles


,primary_id,region,time_period,area_agrc_crops_bevs_and_spices,area_agrc_crops_cereals,area_agrc_crops_fibers,area_agrc_crops_fruits,area_agrc_crops_herbs_and_other_perennial_crops,area_agrc_crops_nuts,area_agrc_crops_other_annual,...,yf_agrc_herbs_and_other_perennial_crops_tonne_ha,yf_agrc_nuts_tonne_ha,yf_agrc_other_annual_tonne_ha,yf_agrc_other_woody_perennial_tonne_ha,yf_agrc_pulses_tonne_ha,yf_agrc_rice_tonne_ha,yf_agrc_sugar_cane_tonne_ha,yf_agrc_tubers_tonne_ha,yf_agrc_vegetables_and_vines_tonne_ha,yf_lndu_supremum_pastures_tonne_per_ha
0,6520974,uganda,4,634960.593061,2.025945e+06,156073.139455,1.131815e+06,505.429083,513832.892470,2.915383e+06,...,2.172065,0.458120,1.528325,0.3674,0.799814,2.970174,77.726965,4.805888,4.035314,92.81
1,6520974,uganda,5,640126.452982,2.042427e+06,157342.906406,1.141024e+06,509.541112,518013.291655,2.939102e+06,...,2.196794,0.611231,1.681739,0.3674,0.805568,3.032345,78.023706,4.440105,4.013004,92.81


---
## 11. Preparar columnas y upload de emisiones totales a S3

In [161]:
decomposed_df['total_emissions'] = decomposed_df.filter(like="emission_co2e_subsector_total").sum(axis=1)

energy_demand_cols    = [c for c in decomposed_df.columns if c.startswith("energy_demand_")]
total_value_enfu_cols = [c for c in decomposed_df.columns if c.startswith("totalvalue_enfu_fuel_consumed_inen")]
frac_inen_energy_cols = [c for c in input_df.columns if c.startswith("frac_inen_energy_")]
efficfactor_cols      = [c for c in input_df.columns if c.startswith("efficfactor_enfu_industrial_energy_fuel")]

print(f"energy_demand        : {len(energy_demand_cols)} cols")
print(f"total_value_enfu     : {len(total_value_enfu_cols)} cols")
print(f"frac_inen_energy     : {len(frac_inen_energy_cols)} cols")
print(f"efficfactor          : {len(efficfactor_cols)} cols")

energy_demand        : 167 cols
total_value_enfu     : 12 cols
frac_inen_energy     : 274 cols
efficfactor          : 13 cols


In [162]:
df_to_upload = pd.merge(
    decomposed_df,
    input_df[["primary_id", "region", "time_period"] + efficfactor_cols + frac_inen_energy_cols],
    on=["primary_id", "region", "time_period"],
    how="left"
)

cols_to_keep = (
    ["primary_id", "time_period", "total_emissions"]
    + efficfactor_cols
    + energy_demand_cols
    + frac_inen_energy_cols
    + total_value_enfu_cols
)
missing_cols = [c for c in cols_to_keep if c not in df_to_upload.columns]
if missing_cols:
    print(f"WARNING: columnas faltantes: {missing_cols[:10]}")

cols_to_keep = [c for c in cols_to_keep if c in df_to_upload.columns]
df_to_upload = df_to_upload[cols_to_keep]
print(f"df_to_upload shape: {df_to_upload.shape}")
df_to_upload.head(2)

df_to_upload shape: (52, 469)


,primary_id,time_period,total_emissions,efficfactor_enfu_industrial_energy_fuel_biomass,efficfactor_enfu_industrial_energy_fuel_coal,efficfactor_enfu_industrial_energy_fuel_coke,efficfactor_enfu_industrial_energy_fuel_diesel,efficfactor_enfu_industrial_energy_fuel_electricity,efficfactor_enfu_industrial_energy_fuel_furnace_gas,efficfactor_enfu_industrial_energy_fuel_gasoline,...,totalvalue_enfu_fuel_consumed_inen_fuel_coke,totalvalue_enfu_fuel_consumed_inen_fuel_diesel,totalvalue_enfu_fuel_consumed_inen_fuel_electricity,totalvalue_enfu_fuel_consumed_inen_fuel_furnace_gas,totalvalue_enfu_fuel_consumed_inen_fuel_gasoline,totalvalue_enfu_fuel_consumed_inen_fuel_hydrocarbon_gas_liquids,totalvalue_enfu_fuel_consumed_inen_fuel_hydrogen,totalvalue_enfu_fuel_consumed_inen_fuel_kerosene,totalvalue_enfu_fuel_consumed_inen_fuel_natural_gas,totalvalue_enfu_fuel_consumed_inen_fuel_oil
0,6520974,4,111.777536,0.6,0.6,0.6,0.75,2.4,0.8,0.75,...,1.002349,141.607364,246.462510,2.391943e+06,158.208748,51.110238,0.0,11.656464,25.072955,19.863422
1,6520974,5,113.675322,0.6,0.6,0.6,0.75,2.4,0.8,0.75,...,1.048178,164.468054,289.547615,2.583035e+06,177.497457,96.572726,0.0,19.167538,29.820597,28.580750


In [163]:
df_to_upload = df_to_upload[df_to_upload['primary_id'] != 0]

In [164]:
df_to_upload.shape

(52, 469)

In [165]:
s3_key = f"{S3_DECOMPOSED_PREFIX}emission_total_{TEST_PRIMARY_ID}.csv"
print(f"Subiendo a: {s3_key}")
try:
    upload_df_to_s3(df_to_upload, s3, BUCKET_NAME, s3_key)
except Exception as e:
    print(f"ERROR: {e}")
    traceback.print_exc()

Subiendo a: run_database/sisepuede_run_2026-03-10t13;27;53.264959/decomposed_outputs/emission_total_6520974.csv
Uploaded -> s3://sisepuede-data/run_database/sisepuede_run_2026-03-10t13;27;53.264959/decomposed_outputs/emission_total_6520974.csv


---
## 12. CBA — append baseline (primary_id=0) + actual, sin renombrar nada

In [166]:
# Cargar baseline desde el pickle separado (no desde disco, igual que los workers)
# baseline_decomposed_df.pkl ya tiene input+output merged, primary_id=0
base_decomposed_df = pd.read_pickle(os.path.join(CACHE_DIR, "baseline_decomposed_df.pkl"))
print(f"Baseline shape      : {base_decomposed_df.shape}")
print(f"primary_ids en base : {base_decomposed_df['primary_id'].unique()}")
if base_decomposed_df.empty:
    print("ERROR: baseline vacío")
base_decomposed_df.head(2)

Baseline shape      : (56, 4054)
primary_ids en base : [0]


,primary_id,region,time_period,area_gnrl_country_ha,area_lndu_infimum_croplands_ha,area_lndu_infimum_flooded_ha,area_lndu_infimum_forests_mangroves_ha,area_lndu_infimum_forests_primary_ha,area_lndu_infimum_forests_secondary_ha,area_lndu_infimum_grasslands_ha,...,yield_agrc_fruits_tonne,yield_agrc_herbs_and_other_perennial_crops_tonne,yield_agrc_nuts_tonne,yield_agrc_other_annual_tonne,yield_agrc_other_woody_perennial_tonne,yield_agrc_pulses_tonne,yield_agrc_rice_tonne,yield_agrc_sugar_cane_tonne,yield_agrc_tubers_tonne,yield_agrc_vegetables_and_vines_tonne
0,0,uganda,0,24155000,-999,3526780,-999,146461.012134,383133.987866,-999,...,8.162759e+06,1049.688971,382983.417459,2.802339e+06,1294.273957,710613.538152,308317.181306,6.763261e+06,7.028016e+06,2.357715e+06
1,0,uganda,1,24155000,-999,3526780,-999,146461.012134,383133.987866,-999,...,8.014876e+06,1077.840167,361439.962580,2.819014e+06,1299.678998,731167.999372,317530.631920,6.850348e+06,7.235665e+06,2.408238e+06


In [167]:
# Obtener future_id, strategy_id, strategy_code para TEST_PRIMARY_ID
try:
    future_id = int(
        attribute_primary_df.loc[attribute_primary_df["primary_id"] == TEST_PRIMARY_ID, "future_id"].values[0]
    )
    strategy_id = int(
        attribute_primary_df.loc[attribute_primary_df["primary_id"] == TEST_PRIMARY_ID, "strategy_id"].values[0]
    )
    strategy_code = attribute_strategy_df.loc[
        attribute_strategy_df["strategy_id"] == strategy_id, "strategy_code"
    ].values[0]
    print(f"future_id={future_id} | strategy_id={strategy_id} | strategy_code={strategy_code}")
except Exception as e:
    print(f"ERROR: {e}")

future_id=322 | strategy_id=6004 | strategy_code=PFLO:HBLE


In [168]:
# Append baseline + actual — primary_id=0 ya es el baseline, sin renombrar
ssp_data = pd.concat([base_decomposed_df, decomposed_df_merged], ignore_index=True)
ssp_data = ssp_data.replace(np.nan, 0.0)

print(f"ssp_data shape       : {ssp_data.shape}")
print(f"primary_ids en concat: {sorted(ssp_data['primary_id'].unique())}")
# Debe mostrar [0, TEST_PRIMARY_ID]
ssp_data.head(2)

ssp_data shape       : (108, 4054)
primary_ids en concat: [np.int64(0), np.int64(6520974)]


,primary_id,region,time_period,area_gnrl_country_ha,area_lndu_infimum_croplands_ha,area_lndu_infimum_flooded_ha,area_lndu_infimum_forests_mangroves_ha,area_lndu_infimum_forests_primary_ha,area_lndu_infimum_forests_secondary_ha,area_lndu_infimum_grasslands_ha,...,yield_agrc_fruits_tonne,yield_agrc_herbs_and_other_perennial_crops_tonne,yield_agrc_nuts_tonne,yield_agrc_other_annual_tonne,yield_agrc_other_woody_perennial_tonne,yield_agrc_pulses_tonne,yield_agrc_rice_tonne,yield_agrc_sugar_cane_tonne,yield_agrc_tubers_tonne,yield_agrc_vegetables_and_vines_tonne
0,0,uganda,0,24155000.0,-999.0,3526780.0,-999.0,146461.012134,383133.987866,-999.0,...,8.162759e+06,1049.688971,382983.417459,2.802339e+06,1294.273957,710613.538152,308317.181306,6.763261e+06,7.028016e+06,2.357715e+06
1,0,uganda,1,24155000.0,-999.0,3526780.0,-999.0,146461.012134,383133.987866,-999.0,...,8.014876e+06,1077.840167,361439.962580,2.819014e+06,1299.678998,731167.999372,317530.631920,6.850348e+06,7.235665e+06,2.408238e+06


In [169]:
from costs_benefits_ssp.cb_calculate import CostBenefits

strategy_code_base = "BASE"
cb = CostBenefits(ssp_data, attribute_primary_df, attribute_strategy_df, strategy_code_base)
cb.ssp_data["future_id"] = 0
print("CostBenefits instanciado OK")

The TX TX:FRST:INCREASE_SEQUESTRATION_NZ is missing on AttTransformationCode
The TX TX:LNDU:DEC_WETLAND_LOSS_NZ is missing on AttTransformationCode
The TX TX:LNDU:SET_WETLANDS_MINIMUM_NZ is missing on AttTransformationCode
The TX TX:SCOE:INC_EFFICIENCY_HEAT_NZ is missing on AttTransformationCode
The TX TX:FRST:INCREASE_SEQUESTRATION_NDC_2 is missing on AttTransformationCode
The TX TX:LNDU:DEC_WETLAND_LOSS_NDC_2 is missing on AttTransformationCode
The TX TX:LNDU:SET_WETLANDS_MINIMUM_NDC_2 is missing on AttTransformationCode
The TX TX:SCOE:INC_EFFICIENCY_HEAT_NDC_2 is missing on AttTransformationCode
The TX TX:FRST:INCREASE_SEQUESTRATION_NDC_25 is missing on AttTransformationCode
The TX TX:LNDU:DEC_WETLAND_LOSS_NDC_25 is missing on AttTransformationCode
The TX TX:LNDU:SET_WETLANDS_MINIMUM_NDC_25 is missing on AttTransformationCode
The TX TX:SCOE:INC_EFFICIENCY_HEAT_NDC_25 is missing on AttTransformationCode
The TX TX:FRST:INCREASE_SEQUESTRATION_NZ is missing on AttTransformationCode
The 

In [170]:
print(f"Cargando CB config: {CB_CONFIG_PATH}")
cb.load_cb_parameters(CB_CONFIG_PATH)
print("Parámetros CB cargados OK")

Cargando CB config: /Users/fabianfuentes/git/ssp_uganda_data/ssp_modeling/100k_runs_postprocessing/config/cb_config_params.xlsx
Cargamos configuración de archivo excel
Se actualizó la base de datos
Parámetros CB cargados OK


In [171]:
print(f"Calculando costos para strategy_code={strategy_code}...")
try:
    results_system = cb.compute_system_cost_for_strategy(strategy_code_tx=strategy_code)
    results_tx     = cb.compute_technical_cost_for_strategy(strategy_code_tx=strategy_code)
    results_all    = pd.concat([results_system, results_tx], ignore_index=True)
    print(f"results_all shape: {results_all.shape}")
    results_all.head(2)
except Exception as e:
    print(f"ERROR: {e}")
    traceback.print_exc()

Calculando costos para strategy_code=PFLO:HBLE...
---------Costs for: cb:wali:technical_cost:sanitation:unimp_rural.
La variable se evalúa en System Cost
---------Costs for: cb:wali:technical_cost:sanitation:imp_rural.
La variable se evalúa en System Cost
---------Costs for: cb:wali:technical_cost:sanitation:safeman_rural.
La variable se evalúa en System Cost
---------Costs for: cb:wali:technical_cost:sanitation:unimp_urban.
La variable se evalúa en System Cost
---------Costs for: cb:wali:technical_cost:sanitation:imp_urban.
La variable se evalúa en System Cost
---------Costs for: cb:wali:technical_cost:sanitation:safeman_urban.
La variable se evalúa en System Cost
---------Costs for: cb:wali:technical_cost:sanitation:omit_rural.
La variable se evalúa en System Cost
---------Costs for: cb:entc:technical_cost:electricity:capex.
La variable se evalúa en System Cost
---------Costs for: cb:entc:technical_cost:electricity:transmission.
La variable se evalúa en System Cost
---------Costs for

/Users/fabianfuentes/anaconda3/envs/ssp_uganda_env/lib/python3.11/site-packages/costs_benefits_ssp/cb_calculate.py:856: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  tmp["difference_variable"] = cb_orm.diff_var
/Users/fabianfuentes/anaconda3/envs/ssp_uganda_env/lib/python3.11/site-packages/costs_benefits_ssp/cb_calculate.py:857: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  tmp["difference_value"] = data_merged["difference"]
/Users/fabianfuentes/anaconda3/envs/ssp_uganda_env/lib/python3.11/site-packages/cos

---------Costs for: cb:fgtv:technical_cost:flaring:X.
La variable se evalúa en Transformation Cost
---------Costs for: cb:fgtv:technical_cost:leaks:X.
La variable se evalúa en Transformation Cost
---------Costs for: cb:waso:technical_cost:consumer_food_waste:X.
La variable se evalúa en Transformation Cost
---------Costs for: cb:waso:consumer_savings:consumer_food_waste:X.
La variable se evalúa en Transformation Cost
---------Costs for: cb:lvst:technical_cost:ent_ferm_mgmt:X.
La variable se evalúa en Transformation Cost
---------Costs for: cb:agrc:technical_cost:rice_mgmt:X.
La variable se evalúa en Transformation Cost
---------Costs for: cb:agrc:technical_cost:producer_food_waste:X.
La variable se evalúa en Transformation Cost
---------Costs for: cb:agrc:technical_savings:producer_food_waste:X.
La variable se evalúa en Transformation Cost
---------Costs for: cb:agrc:technical_cost:increase_productivity:X.
La variable se evalúa en Transformation Cost
---------Costs for: cb:lvst:technica

In [172]:
try:
    results_all_pp         = cb.cb_process_interactions(results_all)
    results_all_pp_shifted = cb.cb_shift_costs(results_all_pp)
    results_all_pp_shifted["primary_id"] = TEST_PRIMARY_ID
    results_all_pp_shifted["future_id"]  = future_id
    print(f"Postprocesado shape: {results_all_pp_shifted.shape}")
    results_all_pp_shifted.head(2)
except Exception as e:
    print(f"ERROR: {e}")
    traceback.print_exc()

Resolving Interactions in SCOE : TX:SCOE:INC_EFFICIENCY_APPLIANCE, TX:SCOE:SHIFT_FUEL_HEAT, TX:SCOE:SHIFT_FUEL_HEAT, TX:SCOE:DEC_DEMAND_HEAT 
Resolving Interactions in INEN : TX:INEN:SHIFT_FUEL_HEAT, TX:INEN:SHIFT_FUEL_HEAT, TX:INEN:SHIFT_FUEL_HEAT, TX:INEN:SHIFT_FUEL_HEAT, TX:INEN:SHIFT_FUEL_HEAT, TX:INEN:SHIFT_FUEL_HEAT, TX:INEN:INC_EFFICIENCY_ENERGY 
Postprocesado shape: (15532, 11)


/Users/fabianfuentes/anaconda3/envs/ssp_uganda_env/lib/python3.11/site-packages/costs_benefits_ssp/cb_calculate.py:660: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  res_pre2025["variable"] = res_pre2025["variable"] + "_shifted" + (res_pre2025["time_period"]+SSP_GLOBAL_TIME_PERIOD_0).astype(str)#create a new variable so they can be recognized as shifted costs
/Users/fabianfuentes/anaconda3/envs/ssp_uganda_env/lib/python3.11/site-packages/costs_benefits_ssp/cb_calculate.py:661: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-

---
## 13. Agregar CBA y subir a S3

In [175]:
def postprocess_cba(cb_raw_df: pd.DataFrame) -> pd.DataFrame:
    parts = cb_raw_df["variable"].astype(str).str.split(":", n=4, expand=True)
    parts.columns = ["name", "sector", "cb_type", "item_1", "item_2"]
    cb_data = pd.concat([cb_raw_df, parts], axis=1)
    cb_data["value"] = cb_data["value"] / 1e9
    cb_data["Year"]  = cb_data["time_period"] + 2015

    group_cols = ["cb_type", "strategy_code", "primary_id", "future_id", "Year"]
    cb_agg = (
        cb_data.groupby(group_cols, dropna=False, as_index=False)["value"]
        .sum()
        .rename(columns={"value": "Cumulative"})
    )
    agg_cb_df = (
        cb_agg.pivot_table(
            index=["primary_id", "future_id", "strategy_code", "Year"],
            columns="cb_type",
            values="Cumulative",
            aggfunc="sum"
        )
        .reset_index()
    )
    agg_cb_df.columns.name = None
    return agg_cb_df

try:
    agg_cb_df = postprocess_cba(results_all_pp_shifted)
    print(f"agg_cb_df shape: {agg_cb_df.shape}")
    agg_cb_df.head()
except Exception as e:
    print(f"ERROR en postprocess_cba: {e}")
    traceback.print_exc()

agg_cb_df shape: (52, 21)


In [177]:
agg_cb_df.tail()

,primary_id,future_id,strategy_code,Year,air_pollution,congestion,consumer_savings,crop_value,ecosystem_services,env_pollution,...,human_health,ippu_value,land_pollution,lvst_value,road_safety,sector_specific,system_cost,technical_cost,technical_savings,water_pollution
47,6520974,322,PFLO:HBLE,2066,0.827332,0.722500,22.435560,-0.727257,0.322579,2.635525,...,27.666981,0.037078,0.015999,-1.743645,0.985862,2.001770,7.676017,-29.087414,5.062635,10.219676
48,6520974,322,PFLO:HBLE,2067,0.847057,0.738979,23.283617,-0.819900,0.324964,2.735747,...,28.802547,0.038631,0.016674,-1.755818,1.008566,2.081141,8.052670,-30.292998,5.264437,11.019843
49,6520974,322,PFLO:HBLE,2068,0.867121,0.755731,24.144649,-0.916961,0.330157,2.837481,...,29.959119,0.040213,0.017362,-1.769655,1.031650,2.162490,8.443395,-32.435421,5.468905,11.862409
50,6520974,322,PFLO:HBLE,2069,0.887615,0.772837,25.018234,-1.018249,0.338285,2.940716,...,31.136275,0.041824,0.018062,-1.789741,1.055222,2.245866,8.848772,-33.317306,5.675653,12.748688
51,6520974,322,PFLO:HBLE,2070,0.908612,0.790360,25.905615,-1.123703,0.349372,3.045556,...,32.335709,0.043465,0.018776,-1.818762,1.079367,2.331316,9.269150,-34.796856,5.885170,13.681152


In [178]:
if agg_cb_df is not None and not agg_cb_df.empty:
    s3_key_cb = f"{S3_CB_PREFIX}cb_{TEST_PRIMARY_ID}.csv"
    print(f"Subiendo CBA -> {s3_key_cb}")
    try:
        upload_df_to_s3(agg_cb_df, s3, BUCKET_NAME, s3_key_cb)
    except Exception as e:
        print(f"ERROR: {e}")
        traceback.print_exc()
else:
    print("agg_cb_df vacío — nada que subir")

Subiendo CBA -> run_database/sisepuede_run_2026-03-10t13;27;53.264959/cb_outputs/cb_6520974.csv
Uploaded -> s3://sisepuede-data/run_database/sisepuede_run_2026-03-10t13;27;53.264959/cb_outputs/cb_6520974.csv


---
## 14. Cleanup (opcional)

Elimina CSVs temporales excepto `uganda_0.csv` y borra el cache.

In [96]:
KEEP_TMP = True  # Cambia a False para limpiar

if not KEEP_TMP:
    deleted = []
    for fn in os.listdir(TMP_DIR):
        if fn.startswith("uganda_") and fn.endswith(".csv") and fn != "uganda_0.csv":
            os.remove(os.path.join(TMP_DIR, fn))
            deleted.append(fn)
    print(f"Eliminados {len(deleted)} archivos")

    if os.path.isdir(CACHE_DIR):
        shutil.rmtree(CACHE_DIR)
        print("tmp/cache/ eliminado")

    # Verificar que uganda_0.csv sigue ahí
    print(f"uganda_0.csv existe: {os.path.exists(BASELINE_CSV_PATH)}")
else:
    print("KEEP_TMP=True — no se eliminó nada")
    print(f"Archivos en tmp/: {[f for f in os.listdir(TMP_DIR) if f.endswith('.csv')]}")

KEEP_TMP=True — no se eliminó nada
Archivos en tmp/: ['total_emissions_by_primary_id.csv', 'uganda_6520974.csv', 'uganda_0.csv']
